# IntrAgent Demo

Run a single question against either a local markdown paper or one example from `IntrAgent/IntraBench`.

In [ ]:
from pathlib import Path
from datasets import load_dataset

from rank import MarkdownPaperProcessor
from agent import LLMChunkProcessor

In [ ]:
config_path = "config.json"
input_mode = "local"  # choose from: "local", "hf"

# Local mode
paper_md_path = "path to .md"
question = "What are the main analytes type studied?"

# Hugging Face mode
hf_dataset_name = "IntrAgent/IntraBench"
hf_split = "train"
hf_row_index = 0

In [ ]:
def _pick_first(record, candidates):
    for key in candidates:
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value
    return None


def load_hf_example(dataset_name, split, row_index):
    dataset = load_dataset(dataset_name, split=split)
    if row_index < 0 or row_index >= len(dataset):
        raise IndexError(f"row_index {row_index} is out of range for split '{split}' with {len(dataset)} rows")

    record = dataset[row_index]
    paper_markdown = _pick_first(record, ["paper_md", "paper_markdown", "markdown", "context", "document", "paper"])
    hf_question = _pick_first(record, ["question", "query", "prompt"])

    if paper_markdown is None:
        raise KeyError(f"Could not find paper markdown text in dataset columns: {list(record.keys())}")
    if hf_question is None:
        raise KeyError(f"Could not find question text in dataset columns: {list(record.keys())}")

    return record, paper_markdown, hf_question

In [ ]:
if input_mode == "local":
    md_path = Path(paper_md_path).expanduser().resolve()
    if not md_path.exists():
        raise FileNotFoundError(f"Markdown file not found: {md_path}")
    if not question.strip():
        raise ValueError("Please provide a non-empty question for local mode.")

    selected_question = question.strip()
    selected_file_path = md_path
    dataset_record = None
    print(f"Using local markdown file: {selected_file_path}")

elif input_mode == "hf":
    dataset_record, paper_markdown, selected_question = load_hf_example(
        hf_dataset_name,
        hf_split,
        hf_row_index,
    )
    hf_cache_dir = Path(".hf_cache")
    hf_cache_dir.mkdir(exist_ok=True)
    selected_file_path = hf_cache_dir / f"{hf_dataset_name.replace('/', '_')}_{hf_split}_{hf_row_index}.md"
    selected_file_path.write_text(paper_markdown, encoding="utf-8")
    print(f"Loaded HF example with columns: {list(dataset_record.keys())}")
    print(f"Materialized markdown to: {selected_file_path}")

else:
    raise ValueError("input_mode must be either 'local' or 'hf'")

print(f"Question: {selected_question}")

In [ ]:
processor = MarkdownPaperProcessor(str(selected_file_path.parent), selected_question, config_path)
result_dict = processor.process_file(str(selected_file_path))
knowledge_chunks = [entry[2] for entry in result_dict["entries"]]

if not knowledge_chunks:
    raise ValueError("No knowledge chunks were produced from the selected markdown file.")

print(f"Prepared {len(knowledge_chunks)} knowledge chunks")

In [ ]:
agent = LLMChunkProcessor(
    config_path=config_path,
    question=selected_question,
    knowledge_chunks=knowledge_chunks,
)
final_answer = agent.answer_question()
final_answer